In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import lxml

In [2]:
headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko)'}
Webpage = requests.get('https://books.toscrape.com/catalogue/page-1.html',headers=headers).text


In [3]:
soup = BeautifulSoup(Webpage,'lxml') #html parsel

In [4]:
print(soup.prettify())

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   All products | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:30" name="created"/>
  <meta content="" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="../static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="../static/oscar/css/styles.css" rel="stylesheet" typ

In [5]:
soup.find_all('h3')[0].text # how to extrct one

'A Light in the ...'

In [6]:
len(soup.find_all('h3'))

20

In [7]:
for i in soup.find_all('h3'):
    print(i.text.strip()) #strip is use to remove special character

A Light in the ...
Tipping the Velvet
Soumission
Sharp Objects
Sapiens: A Brief History ...
The Requiem Red
The Dirty Little Secrets ...
The Coming Woman: A ...
The Boys in the ...
The Black Maria
Starving Hearts (Triangular Trade ...
Shakespeare's Sonnets
Set Me Free
Scott Pilgrim's Precious Little ...
Rip it Up and ...
Our Band Could Be ...
Olio
Mesaerion: The Best Science ...
Libertarianism for Beginners
It's Only the Himalayas


In [8]:
for i in soup.find_all('p', class_='price_color'):
    price_clean = i.text.replace("Â£","£") 
    print(price_clean.strip())

£51.77
£53.74
£50.10
£47.82
£54.23
£22.65
£33.34
£17.93
£22.60
£52.15
£13.99
£20.66
£17.46
£52.29
£35.02
£57.25
£23.88
£37.59
£51.33
£45.17


In [9]:
for i in soup.find_all('p', class_="instock availability"):
    print(i.text.strip())

In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock
In stock


In [10]:
books = soup.find_all('article',class_="product_pod")

In [11]:
len(books)

20

In [12]:
Book_Name = []
Price = []
Star_rating=[]
Available = []

for i in books:
    Book_Name.append(i.find('h3').text.strip())

    clean_p = i.find('p',class_="price_color").text.replace("Â£","£")
    Price.append(clean_p.strip())

    Available.append(i.find('p',class_='instock availability').text.strip())

    rating_tag = i.find('p',class_='star-rating')
    Star_rating.append(rating_tag['class'][1])

d = {"Book_Name":Book_Name ,"Star_rating":Star_rating , "Price":Price , "Availablity": Available}

df = pd.DataFrame(d)


In [13]:
df.head()

,Book_Name,Star_rating,Price,Availablity
0,A Light in the ...,Three,£51.77,In stock
1,Tipping the Velvet,One,£53.74,In stock
2,Soumission,One,£50.10,In stock
3,Sharp Objects,Four,£47.82,In stock
4,Sapiens: A Brief History ...,Five,£54.23,In stock


In [14]:
df.shape

(20, 4)

In [15]:
final = pd.DataFrame()

for j in range(1,51):

    url = 'https://books.toscrape.com/catalogue/page-{}.html'.format(j)
    headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko)'}
    Webpage = requests.get(url,headers=headers).text
    soup = BeautifulSoup(Webpage,'lxml') 
    books = soup.find_all('article',class_="product_pod")

    Book_Name = []
    Price = []
    Star_rating=[]
    Available = []

    for i in books:
        Book_Name.append(i.find('h3').text.strip())

        clean_p = i.find('p',class_="price_color").text.replace("Â£","£")
        Price.append(clean_p.strip())

        Available.append(i.find('p',class_='instock availability').text.strip())

        rating_tag = i.find('p',class_='star-rating')
        Star_rating.append(rating_tag['class'][1])

    d = {"Book_Name":Book_Name ,"Star_rating":Star_rating , "Price":Price , "Availablity": Available}

    df = pd.DataFrame(d)

    final = pd.concat([final,df],ignore_index=True)

In [16]:
final


,Book_Name,Star_rating,Price,Availablity
0,A Light in the ...,Three,£51.77,In stock
1,Tipping the Velvet,One,£53.74,In stock
2,Soumission,One,£50.10,In stock
3,Sharp Objects,Four,£47.82,In stock
4,Sapiens: A Brief History ...,Five,£54.23,In stock
...,...,...,...,...
995,Alice in Wonderland (Alice's ...,One,£55.53,In stock
996,"Ajin: Demi-Human, Volume 1 ...",Four,£57.06,In stock
997,A Spy's Devotion (The ...,Five,£16.97,In stock
998,1st to Die (Women's ...,One,£53.98,In stock


In [19]:
final[final['Price'] == '£26.08']

,Book_Name,Star_rating,Price,Availablity
999,"1,000 Places to See ...",Five,£26.08,In stock


In [21]:
final.to_csv('Books Inventory Dataset.csv',index=False)